# Z24 experiment runner

This notebook contains no model logic. Select one config, then run all cells. The dataset must be attached at `/kaggle/input/dataset`. Run each experiment in a fresh Kaggle session so TensorFlow and PyTorch do not retain each other's GPU memory.

In [ ]:
# Cell 1 - settings you may change
from pathlib import Path

REPO_URL = 'https://github.com/tranvanphuongdevdream-web/shm_ml.git'
BRANCH = 'master'
EXPERIMENT_ID = 'dcnn_002'  # dcnn_001 | dcnn_002 | tsai_001
INPUTS_PATH = Path('/kaggle/input/dataset/inputs.npy')
LABELS_PATH = Path('/kaggle/input/dataset/labels.npy')
OUTPUT_DIR = Path('/kaggle/working/results')
REPO_DIR = Path('/kaggle/working/shm_ml')

assert INPUTS_PATH.is_file(), f'Missing {INPUTS_PATH}'
assert LABELS_PATH.is_file(), f'Missing {LABELS_PATH}'
assert EXPERIMENT_ID in {'dcnn_001', 'dcnn_002', 'tsai_001'}


In [ ]:
# Cell 2 - clone once; later runs update without deleting the directory
import subprocess
import sys

if (REPO_DIR / '.git').is_dir():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH])
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository')
else:
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, str(REPO_DIR)])

print('Repository ready:', REPO_DIR)


In [ ]:
# Cell 3 - one requirements file; keep Kaggle's CUDA-enabled PyTorch
from importlib.metadata import PackageNotFoundError, version

if EXPERIMENT_ID == 'tsai_001':
    requirements = [
        line.strip()
        for line in (REPO_DIR / 'requirements.txt').read_text(encoding='utf-8').splitlines()
        if line.strip() and not line.lstrip().startswith('#')
    ]
    tsai_specs = [spec for spec in requirements if spec.startswith('tsai==')]
    if len(tsai_specs) != 1:
        raise ValueError('requirements.txt must contain exactly one tsai== version')
    other_specs = [spec for spec in requirements if spec != tsai_specs[0]]
    try:
        torch_before = version('torch')
    except PackageNotFoundError:
        torch_before = None
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
        *other_specs,
    ])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
        '--no-deps', tsai_specs[0],
    ])
    assert version('torch') == torch_before, 'The Kaggle PyTorch build changed unexpectedly'
print('Dependencies ready for:', EXPERIMENT_ID)


In [ ]:
# Cell 4 - run in a child process so framework GPU memory is released on exit
command = [
    sys.executable, 'run.py', 'train', '--experiment', EXPERIMENT_ID,
]
print('Running:', ' '.join(command))
subprocess.check_call(command, cwd=REPO_DIR)
print('Finished. Download the ZIP from:', OUTPUT_DIR)
